# Phase 2: Text Preprocessing

This notebook prepares the cleaned mental health text dataset for feature engineering and machine learning. The workflow includes inspecting text quality, repairing character encoding artifacts, normalizing text, tokenizing, removing selected stop words, applying lemmatization, validating the results, and saving a preprocessed dataset.

In [32]:
# Import libraries
from pathlib import Path

import pandas as pd

In [33]:
# Load processed dataset
PROCESSED_DATA_DIR = Path.cwd().parent / "data" / "processed"

input_file = PROCESSED_DATA_DIR / "mental_health_text_dataset.csv"

mental_health_df = pd.read_csv(input_file)

print("Dataset shape:", mental_health_df.shape)
mental_health_df.head()

Dataset shape: (17699, 2)


,Label,TEXT
0,0.0,TIL the movie Starship Troopers was never adap...
1,0.0,What do you call a fat baby?
2,0.0,Two morons are sitting on a fence. The big one...
3,0.0,I covered all my weapons in glue.
4,0.0,Joke I made up: Caveman and a bear walk into a...


In [34]:
# Explore the text

In [35]:
# View random records
mental_health_df.sample(10, random_state=42)

,Label,TEXT
980,0.0,What fruit must get married locally?
5157,0.0,_ara Good to hear that Allah aapko sehat de (Y)
7359,1.0,I cannot survive the dissolution with my loved...
14814,2.0,"I'm a teen. A teen girl.\nI get mood swings, w..."
2805,0.0,"""Here, next to mine"" wasn't the answer i was e..."
7487,1.0,. I want to tell you my story ...
16754,2.0,I'm struggling.\nI'm actually fucking struggli...
11584,2.0,goodbye message
4203,0.0,I can`t really say much because none of it is...
11000,2.0,I just don't feel like living really.


In [36]:
# Review text length
mental_health_df["character_count"] = mental_health_df["TEXT"].str.len()
mental_health_df["word_count"] = mental_health_df["TEXT"].str.split().str.len()
mental_health_df[["character_count", "word_count"]].describe()

,character_count,word_count
count,17699.000000,17699.000000
mean,572.167976,108.001017
std,1075.965746,191.214970
min,1.000000,1.000000
25%,44.000000,9.000000
50%,135.000000,27.000000
75%,719.000000,140.000000
max,25558.000000,4834.000000


In [37]:
# View the shortest records
mental_health_df.nsmallest(20, "word_count")[["Label", "TEXT", "word_count"]]

,Label,TEXT,word_count
58,0.0,Threesome?,1
251,0.0,Ants,1
255,0.0,Botany,1
661,0.0,V,1
714,0.0,Soldiers,1
797,0.0,#NAME?,1
932,0.0,Pythagoras,1
1148,0.0,Pony,1
1267,0.0,Farmer,1
1327,0.0,Infrequently,1


In [38]:
# Inspect text quality

In [39]:
# Confirm missing values
mental_health_df.isnull().sum()

Label              0
TEXT               0
character_count    0
word_count         0
dtype: int64

In [40]:
# Count duplicated text
duplicate_text_count = mental_health_df.duplicated(subset=["TEXT"]).sum()

print("Duplicated TEXT values:", duplicate_text_count)

Duplicated TEXT values: 1834


In [41]:
# Check if identical text has conflicting labels
text_label_counts = mental_health_df.groupby("TEXT")["Label"].nunique()

conflicting_text_count = (text_label_counts > 1).sum()

print(
    "TEXT values assigned to more than one label:",
    conflicting_text_count,
)

TEXT values assigned to more than one label: 4


In [42]:
# View duplicate examples
duplicate_examples = mental_health_df[
    mental_health_df.duplicated(subset=["TEXT"], keep=False)
].sort_values("TEXT")

duplicate_examples.head(20)

,Label,TEXT,character_count,word_count
8681,1.0,.. the story is probably banal to the extreme...,1446,284
8431,1.0,.. the story is probably banal to the extreme...,1446,284
8385,1.0,"... Now there will be a lot of things, but I ...",132,27
8635,1.0,"... Now there will be a lot of things, but I ...",132,27
16686,2.0,This posting has been approved by the moderat...,2019,341
13317,2.0,This posting has been approved by the moderat...,2019,341
8347,1.0,dear readers.,14,2
8597,1.0,dear readers.,14,2
9200,1.0,everybody.,11,1
9336,1.0,everybody.,11,1


In [43]:
print("Exact duplicate rows:", mental_health_df.duplicated().sum())
print("Duplicate TEXT values:", mental_health_df.duplicated(subset=["TEXT"]).sum())

Exact duplicate rows: 1829
Duplicate TEXT values: 1834


In [44]:
mental_health_df.shape

(17699, 4)

In [45]:
# Check whether identical TEXT has different labels
conflicting_labels = mental_health_df.groupby("TEXT")["Label"].nunique()

print("TEXT values with multiple labels:", (conflicting_labels > 1).sum())

TEXT values with multiple labels: 4


In [51]:
# Find TEXT values that have more than one unique label
conflicting_text = mental_health_df.groupby("TEXT")["Label"].nunique()

conflicting_posts = conflicting_text[conflicting_text > 1].index

# Display all records with conflicting labels
mental_health_df[mental_health_df["TEXT"].isin(conflicting_posts)].sort_values(
    ["TEXT", "Label"]
)[["Label", "TEXT", "character_count", "word_count"]]

,Label,TEXT,character_count,word_count
3330,0.0,.,1,1
7681,1.0,.,1,1
15265,2.0,.,1,1
16512,2.0,.,1,1
1873,0.0,Alone.,6,1
11941,2.0,Alone.,6,1
7810,1.0,I hate myself.,14,3
11815,2.0,I hate myself.,14,3
7190,1.0,I'm tired,9,2
11742,2.0,I'm tired,9,2


In [52]:
# Identify text assigned to multiple labels
conflicting_texts = mental_health_df.groupby("TEXT")["Label"].nunique()

conflicting_texts = conflicting_texts[conflicting_texts > 1].index

# Remove all conflicting records
mental_health_df = mental_health_df[
    ~mental_health_df["TEXT"].isin(conflicting_texts)
].copy()

print("Shape after removing conflicting labels:")
print(mental_health_df.shape)

Shape after removing conflicting labels:
(17689, 4)


In [ ]:
# Remove normal duplicates
mental_health_df = mental_health_df.drop_duplicates(subset=["TEXT"]).copy()

print("Shape after removing duplicate text:")
print(mental_health_df.shape)

Shape after removing duplicate text:
(15861, 4)


## Duplicate Assessment Summary

An inspection of duplicate text identified 1,834 duplicated social media posts. Four text values were assigned to multiple target labels, including ".", "Alone.", "I hate myself.", and "I'm tired". Because identical text associated with different labels introduces ambiguity into supervised machine learning, these conflicting records were removed prior to preprocessing.

After removing the conflicting records, duplicate social media posts with identical text and matching labels were removed, resulting in a final dataset of **15,861 unique observations** for text preprocessing.

In [46]:
# Clean text

In [47]:
# Tokenize text

In [48]:
# Remove stop words

In [49]:
# Lemmatize text

In [50]:
# Save preprocessed dataset